# Pipeline Walkthrough

Step-by-step walkthrough of the RAG ingestion pipeline using local example files.
Each cell mirrors what happens inside the pipeline so you can inspect inputs and outputs at every stage.

**Stages covered:**
1. Setup & file discovery
2. MIME detection & format support check
3. Text extraction (parsing) — one example per format
4. Relevance classification
5. Chunking (markdown-aware splitting)
6. Contextual enrichment (Gemini per-chunk context)
7. Summary

The example files in `example_data/` cover: PDF, DOC, DOCX, PPT, XLSX, HTML, JSON, and a junk HTML page.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.utils.config import config, genai_client

print(f"Project:          {config.project_id}")
print(f"Region:           {config.region}")
print(f"Extraction model: {config.markdown_converter_gemini_model}")
print(f"Relevance model:  {config.relevance_gemini_model}")
print(f"Enrichment model: {config.contextual_chunking_gemini_model}")
print(f"Chunk size:       {config.chunk_size}")
print(f"Chunk overlap:    {config.chunk_overlap}")

## 2. Discover example files

In production, files come from a GCS bucket. Here we use local files from `example_data/`.

The pipeline uses `file_id = MD5(gcs_uri)` as a stable identifier. We simulate that with the file path.

In [ ]:
import hashlib

from src.document_preprocessing.parser import PARSEABLE_MIMES
from src.utils.llm_utils import get_mime_type_from_path

EXAMPLE_DIR = PROJECT_ROOT / "example_data"

files = []
for p in sorted(EXAMPLE_DIR.iterdir()):
    if p.name.startswith("."):
        continue
    mime = get_mime_type_from_path(str(p))
    file_id = hashlib.md5(str(p).encode()).hexdigest()
    supported = mime in PARSEABLE_MIMES or mime in (
        "text/html",
        "text/plain",
        "text/markdown",
    )
    files.append(
        {
            "file_id": file_id,
            "path": str(p),
            "name": p.name,
            "ext": p.suffix,
            "mime": mime,
            "size_kb": p.stat().st_size / 1024,
            "supported": supported,
        }
    )

# Show what we found
print(f"{'File':<75} {'Ext':>5} {'Size':>8} {'MIME':<60} {'OK'}")
print("=" * 160)
for f in files:
    ok = "yes" if f["supported"] else "NO"
    print(f"{f['name']:<75} {f['ext']:>5} {f['size_kb']:>7.0f}K {f['mime']:<60} {ok}")

## 3. Text extraction (parsing)

This is what the **preprocess Cloud Run service** does for each file:
download from GCS, detect format, and extract text as markdown.

Different formats use different extraction paths:

| Format | Method |
|--------|--------|
| PDF | pymupdf page split + Gemini multimodal per page |
| DOCX | LibreOffice → PDF → Gemini multimodal |
| DOC | LibreOffice → PDF → Gemini multimodal |
| PPT | LibreOffice → PPTX → LibreOffice → PDF → Gemini |
| XLSX | openpyxl → markdown tables (no LLM) |
| HTML | BeautifulSoup + Gemini Flash markdown conversion |
| JSON | Pure Python recursive renderer → markdown (no LLM) |

Let's parse each file and see what comes out.

In [ ]:
from src.document_preprocessing.parser import parse, quick_raw_text

for f in files:
    if not f["supported"]:
        f["content"] = ""
        f["error"] = "unsupported format"
        continue

    print(f"\n{'=' * 70}")
    print(f"Parsing: {f['name']}")
    print(f"Format:  {f['ext']} ({f['mime']})")
    print(f"{'=' * 70}")

    # Step 1: Try quick raw text (cheap, no LLM)
    raw = quick_raw_text(f["path"])
    if raw is not None:
        print(f"Quick raw text: {len(raw)} chars")
        if raw.strip():
            print(f"  Preview: {raw.strip()[:150]}...")
    else:
        print("Quick raw text: not available for this format")

    # Step 2: Full extraction
    try:
        text = parse(f["path"], genai_client=genai_client)
        f["content"] = text
        f["error"] = None
        print(f"\nExtracted: {len(text)} chars")
        print("\n--- First 400 chars ---")
        print(text[:400])
    except Exception as e:
        f["content"] = ""
        f["error"] = str(e)
        print(f"\nEXTRACTION FAILED: {e}")

### Zoom in: PDF extraction

PDFs go through a multi-step process:
1. Split into single-page PDFs with pymupdf
2. Extract raw text per page (used as hint for Gemini)
3. Send each page as an image to Gemini multimodal for markdown conversion

Let's see the raw text vs the Gemini-converted markdown for the first PDF.

In [ ]:
import pymupdf

pdf_file = next((f for f in files if f["ext"] == ".pdf"), None)
if pdf_file:
    doc = pymupdf.open(pdf_file["path"])
    print(f"PDF: {pdf_file['name']}")
    print(f"Pages: {len(doc)}\n")

    for i, page in enumerate(doc):
        raw = page.get_text()
        print(f"--- Page {i} raw text ({len(raw)} chars) ---")
        print(raw[:300] if raw.strip() else "[EMPTY — scanned/image-only page]")
        print()
    doc.close()

    print("\n--- Gemini-converted markdown (full output shown above in Step 3) ---")
    print(f"Total output: {len(pdf_file['content'])} chars")
else:
    print("No PDF in example files")

### Zoom in: JSON rendering

JSON files are converted to markdown without any LLM call.
The parser recursively walks the structure: objects become `**key**: value` lines,
arrays of objects become `##`-headed sections separated by `---`.

This works for any schema — product catalogs, API responses, config dumps.

In [ ]:
json_file = next((f for f in files if f["ext"] == ".json"), None)
if json_file and json_file["content"]:
    print(f"JSON: {json_file['name']}")
    print(f"Raw size: {json_file['size_kb']:.0f} KB")
    print(f"Markdown output: {len(json_file['content'])} chars")
    print("\n--- First 800 chars of markdown ---")
    print(json_file["content"][:800])
else:
    print("No JSON in example files")

### Zoom in: junk HTML detection

The pipeline handles junk files gracefully. Empty/login/navigation-only pages
get extracted but then classified as irrelevant in the next step.

The small `.htm` file is a video player wrapper — no useful text content.

In [ ]:
htm_file = next((f for f in files if f["ext"] == ".htm"), None)
if htm_file:
    print(f"File: {htm_file['name']}")
    print(f"Size: {htm_file['size_kb']:.0f} KB")
    print(f"Content length: {len(htm_file.get('content', ''))} chars")
    print("\n--- Full content ---")
    print(htm_file.get("content", "[no content]")[:500])
    print("\n>>> This will be classified as IRRELEVANT in the next step.")
else:
    print("No .htm file in examples")

## 4. Relevance classification

After extraction, the pipeline asks a lightweight Gemini model whether the content
belongs in the knowledge base. This filters out empty pages, login forms,
cookie banners, navigation-only HTML, etc.

- Content < 50 chars → automatically irrelevant
- Classifier failure → defaults to relevant (don't lose data)

In [ ]:
from src.document_preprocessing.document_relevance_classifier import classify_relevance

print(f"{'File':<75} {'Chars':>7} {'Relevant':>10}")
print("=" * 100)

for f in files:
    content = f.get("content", "")
    if not content:
        f["relevant"] = False
        print(f"{f['name']:<75} {0:>7} {'skip':>10}  (no content)")
        continue

    try:
        relevant = classify_relevance(content, genai_client)
        f["relevant"] = relevant
        label = "YES" if relevant else "no"
        print(f"{f['name']:<75} {len(content):>7} {label:>10}")
    except Exception:
        f["relevant"] = True
        print(
            f"{f['name']:<75} {len(content):>7} {'YES':>10}  (classifier error, default=True)"
        )

## 5. Chunking

Relevant documents are split into overlapping chunks using LangChain's `MarkdownTextSplitter`.
It respects markdown structure: prefers to break at headers, then paragraphs, then lines.

The overlap ensures context continuity across chunk boundaries — a sentence at the end
of chunk N also appears at the start of chunk N+1.

In [ ]:
from langchain_text_splitters import MarkdownTextSplitter

splitter = MarkdownTextSplitter(
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap,
)

for f in files:
    if not f.get("relevant"):
        f["chunks"] = []
        continue

    chunks = splitter.split_text(f["content"])
    f["chunks"] = chunks

    print(f"\n{'=' * 70}")
    print(f"{f['name']}")
    print(f"Content: {len(f['content'])} chars → {len(chunks)} chunks")
    print(f"{'=' * 70}")

    for i, chunk in enumerate(chunks):
        print(f"  Chunk {i:2d}: {len(chunk):5d} chars | {chunk[:80]}...")

## 6. Contextual enrichment

This is the key quality step. For each chunk, Gemini generates up to 4 sentences
explaining what the chunk is about relative to the full document.

The context is later **prepended** to the chunk text before embedding, so the
vector captures the chunk's meaning within the document (not just the raw words).

For long documents (> 500K chars), the pipeline uses a sliding window instead of
the full document — each chunk gets the 50K chars before and after its position.

Let's enrich the chunks of one file to see it in action.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

CONTEXTUAL_SYSTEM_PROMPT = """\
You are an expert in document understanding.

**Input**
- A chunk of text extracted from a larger document
- The full document the chunk comes from

**Your task**
Generate up to 4 sentences of context that explain what this chunk is about \
based on the overall document. The goal is that someone reading just the chunk \
plus your context can understand what the chunk is talking about in relation to \
the full document.

Rules:
- Do NOT repeat information already present in the chunk
- Summarize what comes before and after the chunk to situate it
- Ignore document metadata like authors, dates, copyright, contact information
- Write in the same language as the chunk
- Output only the context sentences, no preamble or explanation"""

MAX_FULL_DOC_CHARS = 500_000
WINDOW_PER_SIDE_CHARS = 50_000

# Pick the first file that has chunks
demo_file = next((f for f in files if f.get("chunks")), None)

if demo_file:
    doc_text = demo_file["content"]
    chunks = demo_file["chunks"]
    print(f"Enriching: {demo_file['name']} ({len(chunks)} chunks)\n")

    # Locate each chunk's position in the original text
    chunk_starts = []
    cursor = 0
    for chunk in chunks:
        idx = doc_text.find(chunk, cursor)
        if idx < 0:
            idx = doc_text.find(chunk)
            if idx < 0:
                idx = cursor
        chunk_starts.append(idx)
        cursor = idx + len(chunk) - config.chunk_overlap

    def build_context(text, chunk, chunk_start):
        if len(text) <= MAX_FULL_DOC_CHARS:
            return text
        start = max(0, chunk_start - WINDOW_PER_SIDE_CHARS)
        end = min(len(text), chunk_start + len(chunk) + WINDOW_PER_SIDE_CHARS)
        return text[start:end]

    def enrich_one(idx):
        chunk = chunks[idx]
        doc_context = build_context(doc_text, chunk, chunk_starts[idx])
        if doc_context.strip() == chunk.strip():
            return idx, ""
        resp = genai_client.models.generate_content(
            model=config.contextual_chunking_gemini_model,
            contents=(
                f"DOCUMENT:\n```\n{doc_context}\n```\n\n"
                f"CHUNK:\n```\n{chunk}\n```\n\n"
                "Generate the context useful to understand the chunk."
            ),
            config={"temperature": 0, "system_instruction": CONTEXTUAL_SYSTEM_PROMPT},
        )
        return idx, resp.text or ""

    enriched = [None] * len(chunks)
    with ThreadPoolExecutor(max_workers=8) as pool:
        futs = {pool.submit(enrich_one, i): i for i in range(len(chunks))}
        for fut in as_completed(futs):
            idx, ctx = fut.result()
            enriched[idx] = ctx

    demo_file["enriched"] = enriched

    # Show results
    for i in range(len(chunks)):
        print(f"--- Chunk {i} ---")
        print(
            f"Context:    {enriched[i][:200]}{'...' if len(enriched[i]) > 200 else ''}"
        )
        print(f"Chunk text: {chunks[i][:120]}...")
        print()
else:
    print("No files with chunks to enrich")

### What gets indexed into Vector Search

In production, each chunk is pushed to VS2 as a data object. The `chunk_text` field
contains the context prepended to the chunk — this is what gets embedded and searched.

Let's see what the final indexed text looks like for the first few chunks.

In [ ]:
if demo_file and demo_file.get("enriched"):
    chunks = demo_file["chunks"]
    enriched = demo_file["enriched"]
    file_id = demo_file["file_id"]

    print(f"File: {demo_file['name']}")
    print(f"file_id: {file_id}\n")

    for i in range(min(3, len(chunks))):
        chunk_id = f"{file_id}__{i}"
        ctx = enriched[i]
        # Context prepended to chunk text (same as production)
        indexed_text = f"{ctx}\n\n{chunks[i]}" if ctx.strip() else chunks[i]

        print(f"{'=' * 70}")
        print(f"chunk_id: {chunk_id}")
        print(f"chunk_index: {i}")
        print(f"indexed text ({len(indexed_text)} chars):")
        print(f"{'=' * 70}")
        print(indexed_text[:500])
        if len(indexed_text) > 500:
            print(f"... ({len(indexed_text) - 500} more chars)")
        print()

## 7. Summary

Overview of each file's journey through the pipeline.

In [ ]:
print(f"{'File':<60} {'Ext':>5} {'Chars':>7} {'Relevant':>9} {'Chunks':>7}")
print("=" * 95)
for f in files:
    name = f["name"][:58]
    ext = f["ext"]
    content_len = len(f.get("content", ""))
    relevant = f.get("relevant", False)
    n_chunks = len(f.get("chunks", []))
    error = f.get("error", "")

    r_label = "yes" if relevant else "no"
    if error:
        r_label = f"err: {error[:20]}"
    print(f"{name:<60} {ext:>5} {content_len:>7} {r_label:>9} {n_chunks:>7}")

total_chunks = sum(len(f.get("chunks", [])) for f in files)
relevant_count = sum(1 for f in files if f.get("relevant"))
print(f"\nTotal: {len(files)} files, {relevant_count} relevant, {total_chunks} chunks")

## What happens next (in production)

The steps above run inside two Cloud Run services, orchestrated by a Vertex AI Pipeline:

1. **Preprocess service** — steps 3-4 (extraction + classification), one HTTP request per file
2. **Chunk-index service** — steps 5-6 (chunking + enrichment + VS2 push), one HTTP request per file

The orchestrator dispatches 200 parallel requests to each service and streams results
into BigQuery staging tables. After all files are processed, a MERGE/INSERT finalizes
the data.

At query time, the search performs:
1. **Hybrid search** on the chunks collection (semantic embeddings + BM25 keyword matching)
2. **Reciprocal Rank Fusion** to combine both rankings (70% semantic, 30% text by default)
3. **Full document fetch** from the documents KV collection for each unique file_id hit

See [architecture.md](architecture.md) for the full system design.